# 🤖 Supervised Learning — Learn How Machine Learning Really Works

**A beginner-friendly, step-by-step Google Colab notebook**

In this notebook you will:
1. Understand *what* supervised learning is (with simple real-life examples)
2. Build a **Classification** model (predict a *category*) — Iris flowers 🌸
3. See what "learning" actually means *inside* the machine ⚙️
4. Build a **Regression** model (predict a *number*) — House prices 🏠
5. Learn how to **measure** whether a model is good or bad

> 💡 **How to use this notebook:** Run each code cell one by one using **Shift + Enter**.
> Read the explanation *above* each cell first, then run the code, then look at the output.

## 🧠 Part 0 — What is Supervised Learning?

Imagine you are preparing for an exam using **solved question papers**:
- You see a question **and** its correct answer.
- You practise many such pairs.
- On exam day, you get a **new question** and answer it using what you learned.

**Supervised learning works exactly like this.** We show the computer many examples *with the correct answers*, and it learns the pattern. Then we give it new data and ask it to predict the answer.

| Term | Meaning | Iris flower example | House example |
|---|---|---|---|
| **Features (X)** | The inputs / clues | petal length, petal width… | area, bedrooms, age |
| **Label / Target (y)** | The correct answer we want to predict | species name | price |
| **Model** | The "brain" that learns the pattern | KNN, Decision Tree… | Linear Regression |
| **Training** | Showing the model examples **with** answers | `model.fit(X_train, y_train)` | same |
| **Prediction** | Asking the model about **new** data | `model.predict(X_new)` | same |

### Two types of supervised learning

| Type | Predicts | Examples |
|---|---|---|
| **Classification** | A **category** (a label) | Spam / Not spam, Pass / Fail, Flower species |
| **Regression** | A **number** | House price, Temperature, Marks |

### The ML workflow (we will follow these steps)

```
Load data → Explore → Split (train/test) → Prepare → Train → Predict → Evaluate → Improve
```

## 📦 Step 1 — Import the libraries

Libraries are ready-made toolboxes so we don't write everything from scratch.

- **numpy** → fast maths on numbers
- **pandas** → tables of data (like Excel inside Python)
- **matplotlib / seaborn** → drawing graphs
- **scikit-learn (sklearn)** → the main ML library with datasets, models and tools

Google Colab already has all of these installed, so we only need to `import` them.

In [ ]:
# --- Basic tools ---
import numpy as np                 # numpy  -> maths on numbers and arrays
import pandas as pd                # pandas -> tables (DataFrames)
import matplotlib.pyplot as plt    # matplotlib -> draw graphs
import seaborn as sns              # seaborn -> prettier graphs (built on matplotlib)

# --- Data and preparation tools from scikit-learn ---
from sklearn.datasets import load_iris                  # ready-made flower dataset
from sklearn.model_selection import train_test_split    # splits data into practice set and exam set
from sklearn.preprocessing import StandardScaler        # brings all columns to the same scale

# --- Models (the "brains") ---
from sklearn.neighbors import KNeighborsClassifier      # K-Nearest Neighbours
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC                             # Support Vector Machine

# --- Tools to measure how good a model is ---
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)

sns.set_theme(style="whitegrid")   # make every graph look clean
print("✅ All libraries loaded successfully!")

# 🌸 PART 1 — CLASSIFICATION
### Goal: Look at a flower's measurements and predict its **species**

The **Iris dataset** is the "Hello World" of machine learning. A botanist measured 150 iris flowers of 3 species:
**setosa**, **versicolor** and **virginica**.

## 📥 Step 2 — Load the data

`load_iris(as_frame=True)` gives us the dataset as a pandas table.
- `iris.frame` → the full table (features + target)
- `target` column holds the answer as numbers: `0`, `1`, `2`
- We add a readable `species` column so humans can understand the numbers.

> In real projects you will usually load your own file using `pd.read_csv("file.csv")`.

In [ ]:
iris = load_iris(as_frame=True)        # load the dataset as a pandas table
df = iris.frame                        # df = DataFrame (our table)
class_names = iris.target_names        # ['setosa', 'versicolor', 'virginica']

# Convert target numbers (0,1,2) into readable names
df["species"] = df["target"].map(dict(enumerate(class_names)))

df.head()    # show the first 5 rows

## 🔍 Step 3 — Explore the data (never skip this!)

Before building any model, **understand your data**. We check:
- `shape` → how many rows and columns?
- `isnull().sum()` → are there missing values?
- `value_counts()` → are all classes equally represented?
- `describe()` → min, max, mean of every column

In [ ]:
print("Shape (rows, columns):", df.shape)

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nHow many flowers of each species?")
print(df["species"].value_counts())

df.describe().round(2)   # statistics: count, mean, std, min, max...

## 📊 Step 4 — Visualise the data

A picture tells us if a model *can* learn the pattern. The **pair plot** draws every feature against every other feature, coloured by species.

👀 **What to look for:** Notice how **setosa** (one colour) is completely separated from the others in the petal graphs. That means the features contain a clear pattern — a model will be able to learn it.

In [ ]:
sns.pairplot(df, vars=iris.feature_names, hue="species", height=2.2)
plt.show()

## ✂️ Step 5 — Separate Features (X) and Label (y)

- **X** = the inputs (the 4 measurements)
- **y** = the answer we want to predict (`target`)

Convention: capital **X** (a table) and small **y** (a single column).

In [ ]:
X = df[iris.feature_names]    # all 4 measurement columns
y = df["target"]              # the answer column

print("X (features) looks like:")
print(X.head(3))
print("\ny (label) looks like:")
print(y.head(3))

## 🧪 Step 6 — Split into Training set and Test set

**Why?** If a student sees the *exact* exam questions while practising, a high score proves nothing — they may have just memorised the answers.

So we hide some data from the model:
- **Training set (80%)** → the model learns from this
- **Test set (20%)** → the model has **never seen** this; we use it as the final exam

Parameters:
- `test_size=0.2` → 20% goes to the test set
- `random_state=42` → makes the shuffle repeatable, so everyone gets the same result
- `stratify=y` → keeps the same proportion of each species in both sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training flowers:", X_train.shape[0])
print("Test flowers    :", X_test.shape[0])

## ⚖️ Step 7 — Feature Scaling

Some models (like KNN, which measures *distances*) get confused when one column has big numbers and another has small numbers.
**StandardScaler** converts every column so that **mean = 0** and **standard deviation = 1**.

⚠️ **Very important rule:**
- `fit_transform()` on the **training** data → learn the scale *and* apply it
- `transform()` on the **test** data → only apply the scale learned from training

If we let the scaler look at test data, information "leaks" from the exam into the practice. This mistake is called **data leakage**.

In [ ]:
scaler = StandardScaler()                           # create the scaler

X_train_scaled = scaler.fit_transform(X_train)      # learn from training data + apply
X_test_scaled  = scaler.transform(X_test)           # only apply to test data

# Check: after scaling, mean should be ~0 and std ~1
pd.DataFrame(X_train_scaled, columns=X.columns).describe().round(2).loc[["mean", "std"]]

## 🏋️ Step 8 — Train the model (K-Nearest Neighbours)

**KNN idea — "tell me who your neighbours are, and I'll tell you who you are":**

To classify a new flower, KNN finds the **K most similar flowers** in the training data and takes a **majority vote**.

> Example with K = 5: if 4 of the 5 nearest flowers are *virginica* and 1 is *versicolor* → prediction = **virginica**.

- `n_neighbors=5` → look at 5 neighbours (this is a **hyperparameter**: a setting *we* choose)
- `.fit(X_train, y_train)` → **this is the training step**. (For KNN, "learning" simply means storing the training data.)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)   # create the model (not trained yet)

knn.fit(X_train_scaled, y_train)            # TRAIN: show it the examples with answers

print("✅ Model trained!")

## 🔮 Step 9 — Make predictions on the test set

`.predict()` gives the model the **test flowers (without answers)**, and the model guesses their species.
Then we put **Actual** vs **Predicted** side by side to compare.

In [ ]:
y_pred = knn.predict(X_test_scaled)     # model's guesses for the test flowers

results = pd.DataFrame({
    "Actual":    [class_names[i] for i in y_test],     # the true answers
    "Predicted": [class_names[i] for i in y_pred],     # the model's guesses
})
results["Correct?"] = np.where(results["Actual"] == results["Predicted"], "✅", "❌")

results.head(10)

## 📏 Step 10 — Evaluate: how good is the model?

### 1️⃣ Accuracy
`Accuracy = correct predictions ÷ total predictions`

### 2️⃣ Confusion Matrix
A table showing **where** the model made mistakes.
- **Rows** = actual species, **Columns** = predicted species
- Numbers on the **diagonal** (top-left to bottom-right) = correct predictions
- Numbers **off the diagonal** = mistakes

### 3️⃣ Classification Report
- **Precision** → *When the model says "virginica", how often is it right?*
- **Recall** → *Out of all real virginica flowers, how many did the model find?*
- **F1-score** → a balance of precision and recall (1.0 is perfect)
- **Support** → number of test samples of that class

In [ ]:
# 1) Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2%}  ({(y_pred == y_test).sum()} correct out of {len(y_test)})")

# 2) Confusion matrix as a heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted species")
plt.ylabel("Actual species")
plt.title("Confusion Matrix")
plt.show()

# 3) Precision / Recall / F1
print(classification_report(y_test, y_pred, target_names=class_names))

## 🎛️ Step 11 — Choosing the best K (Overfitting vs Underfitting)

Is K = 5 the best choice? Let's try **K from 1 to 30** and compare:
- **Training accuracy** → score on data the model *has seen*
- **Test accuracy** → score on data the model has *never seen* (this is what really matters)

### Key ideas
- **Overfitting** 😵 → model *memorises* training data, but fails on new data (training score high, test score lower).
- **Underfitting** 😴 → model is too simple and misses the pattern (both scores low).

👀 **Notice:** at **K = 1**, training accuracy is always 100% — every point's nearest neighbour is *itself*, so the model just memorises. A good model has a **good test score**, not just a good training score.

👀 **Also notice:** when **K becomes very large** (around 24 or more), the test accuracy *drops*. The model now asks too many neighbours and becomes too simple → **underfitting**.

✅ The sweet spot is a **middle value of K** where the test accuracy is high and stable.

> The Iris dataset is very easy, so differences look small. On harder real-world data, the gap between train and test is much bigger.

In [ ]:
k_values = range(1, 31)
train_scores = []
test_scores = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)       # new model with this k
    model.fit(X_train_scaled, y_train)                # train it
    train_scores.append(model.score(X_train_scaled, y_train))   # accuracy on training data
    test_scores.append(model.score(X_test_scaled, y_test))      # accuracy on test data

plt.figure(figsize=(8, 4))
plt.plot(k_values, train_scores, marker="o", label="Training accuracy")
plt.plot(k_values, test_scores, marker="s", label="Test accuracy")
plt.xlabel("K (number of neighbours)")
plt.ylabel("Accuracy")
plt.title("Effect of K on accuracy")
plt.legend()
plt.show()

best_score = max(test_scores)
best_ks = [k for k, sc in zip(k_values, test_scores) if sc == best_score]
print(f"Highest test accuracy = {best_score:.1%}, reached by K = {best_ks}")
print("👉 Prefer a K from the middle of the stable range (not K=1), because it is less sensitive to noise.")

## 🥊 Step 12 — Try other models (they all work the same way!)

Here is the beautiful thing about scikit-learn: **every model uses the same 3 steps**

```python
model = SomeModel()          # 1. create
model.fit(X_train, y_train)  # 2. train
model.predict(X_test)        # 3. predict
```

So we can put several models in a loop and compare them fairly:

| Model | Simple idea |
|---|---|
| **KNN** | Vote of the nearest neighbours |
| **Logistic Regression** | Draws a boundary line/curve between classes (despite the name, it's for classification!) |
| **Decision Tree** | A flowchart of Yes/No questions |
| **Random Forest** | Many decision trees voting together |
| **SVM** | Finds the widest "street" separating the classes |

In [ ]:
models = {
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(random_state=42),
    "SVM":                 SVC(),
}

scores = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)                      # train
    scores.append({
        "Model": name,
        "Train accuracy": model.score(X_train_scaled, y_train),
        "Test accuracy":  model.score(X_test_scaled, y_test),
    })

score_table = pd.DataFrame(scores).sort_values("Test accuracy", ascending=False)
display(score_table.round(3))

score_table.set_index("Model")[["Train accuracy", "Test accuracy"]].plot(kind="bar", figsize=(9, 4))
plt.ylim(0.8, 1.05)
plt.title("Model comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.show()

## 🌳 Step 13 — Peek inside a Decision Tree (see how it "thinks")

A decision tree is easy to understand because we can **draw its questions**.
Each box shows:
- the question (e.g. `petal width <= 0.8`)
- `gini` → how mixed the classes are (0 = perfectly pure)
- `samples` → how many flowers reached this box
- `class` → the majority class there

We limit `max_depth=3` so the tree stays small and readable. Trees don't need scaling, so we use the **original** (unscaled) data and the thresholds are in real centimetres.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)          # note: unscaled data is fine for trees

plt.figure(figsize=(14, 7))
plot_tree(tree,
          feature_names=iris.feature_names,
          class_names=list(class_names),
          filled=True, rounded=True, fontsize=9)
plt.show()

print(f"Tree test accuracy: {tree.score(X_test, y_test):.2%}")

## 🌼 Step 14 — Predict a brand-new flower

This is the real purpose of ML: use the trained model on data it has **never seen** — like in a real app.

Remember: new data must go through the **same scaler** that we used for training!

In [ ]:
# A new flower measured by us: sepal length, sepal width, petal length, petal width (cm)
new_flower = pd.DataFrame([[5.9, 3.0, 5.1, 1.8]], columns=X.columns)

new_scaled = scaler.transform(new_flower)           # same scaling as training data
prediction = knn.predict(new_scaled)[0]             # predicted class number
probabilities = knn.predict_proba(new_scaled)[0]    # how sure is the model?

print("Predicted species:", class_names[prediction])
print("\nConfidence for each species:")
for name, p in zip(class_names, probabilities):
    print(f"  {name:<12} {p:.0%}")

# ⚙️ PART 2 — What does "Learning" actually mean?

Before we do regression, let's open the black box.

**Training = finding the numbers (parameters) that make the model's mistakes as small as possible.**

Mini example: a student's **marks** depend on **hours studied**. Our model is just a line:

> **marks = w × hours**   (`w` is the slope — the number the model must *learn*)

- If `w` is wrong → predictions are far from reality → **error is high**
- The best `w` → **error is lowest**

We measure error with **MSE (Mean Squared Error)** = the average of (actual − predicted)².

In [ ]:
# Tiny dataset: hours studied -> marks scored   (roughly marks = 10 x hours)
hours = np.array([1, 2, 3, 4, 5, 6, 7, 8])
marks = np.array([12, 22, 33, 41, 52, 61, 73, 80])

# Try 200 different values of w from 0 to 20 and measure the error of each
slopes = np.linspace(0, 20, 200)
errors = []
for w in slopes:
    predictions = w * hours                        # model's guess using this w
    mse = np.mean((marks - predictions) ** 2)      # how wrong is it on average?
    errors.append(mse)

best_w = slopes[np.argmin(errors)]                 # the w with the smallest error

plt.figure(figsize=(7, 4))
plt.plot(slopes, errors)
plt.scatter(best_w, min(errors), color="red", zorder=5, label=f"Best w ≈ {best_w:.2f}")
plt.xlabel("w (slope)")
plt.ylabel("Error (MSE)")
plt.title("Error is lowest at the best w — that's what 'training' finds")
plt.legend()
plt.show()

### 🏔️ How does the computer find the best `w` without trying everything? — **Gradient Descent**

Imagine you are on a hill in thick fog and want to reach the valley. You can't see the valley, but you can feel the slope under your feet. So you:
1. Feel which way is **downhill**
2. Take a **small step** that way
3. Repeat

That's gradient descent! Each loop:
- `gradient` → the slope of the error (which direction is downhill)
- `learning_rate` → size of each step (too big = you overshoot, too small = very slow)
- `w = w - learning_rate × gradient` → take the step

Watch `w` move towards ~10 and the loss shrink 👇

In [ ]:
w = 0.0                  # start with a bad guess
learning_rate = 0.01     # size of each step

for step in range(1, 21):
    predictions = w * hours
    error = predictions - marks                        # how wrong are we?
    loss = np.mean(error ** 2)                         # MSE
    gradient = 2 * np.mean(error * hours)              # slope of the error w.r.t. w
    w = w - learning_rate * gradient                   # step downhill

    if step in (1, 2, 3, 5, 10, 15, 20):
        print(f"Step {step:>2} | w = {w:6.3f} | loss = {loss:8.2f}")

print(f"\n🎯 Learned w ≈ {w:.2f}  (so: marks ≈ {w:.1f} × hours)")

✅ **This is the heart of ML:** make a guess → measure the error → adjust → repeat.
Bigger models (even deep neural networks) do the same thing, just with millions of numbers instead of one.

# 🏠 PART 3 — REGRESSION
### Goal: Predict the **price** of a house (a number) from its details

Regression predicts a **continuous number**, so we can't use "accuracy" any more. We measure **how far** predictions are from reality.

## 🏗️ Step 15 — Create the dataset

We generate 200 houses with a **secret formula** + some random noise (real life is never perfect!):

> price = 5,00,000 + 3,000 × area + 2,00,000 × bedrooms − 15,000 × age + noise

Later we'll check if the model can **discover this formula on its own** from the data alone.

In [ ]:
rng = np.random.default_rng(42)       # random generator (seed 42 -> same data every time)
n = 200                               # number of houses

area     = rng.uniform(500, 3000, n)  # area in sq.ft
bedrooms = rng.integers(1, 6, n)      # 1 to 5 bedrooms
age      = rng.integers(0, 30, n)     # age of the house in years
noise    = rng.normal(0, 150000, n)   # random variation (+/- ~1.5 lakh)

price = 500000 + 3000 * area + 200000 * bedrooms - 15000 * age + noise

house = pd.DataFrame({
    "area_sqft": area.round(0),
    "bedrooms": bedrooms,
    "age_years": age,
    "price": price.round(0),
})
house.head()

## 🔍 Step 16 — Explore and visualise

Let's see how **area** relates to **price**. If the dots form a rising band, a straight line can describe it.
The **correlation** table gives a number between −1 and +1:
- close to **+1** → both rise together
- close to **−1** → one rises, the other falls
- close to **0** → no relationship

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=house, x="area_sqft", y="price", hue="bedrooms", palette="viridis")
plt.title("Area vs Price")
plt.show()

house.corr().round(2)    # correlation between every pair of columns

## ✂️ Step 17 — Split into train and test

Same idea as before: the model learns from 80% of the houses, and we test it on 20% it has never seen.
(Linear Regression doesn't need feature scaling, so we skip that step.)

In [ ]:
X_h = house[["area_sqft", "bedrooms", "age_years"]]    # features
y_h = house["price"]                                   # target (a number now)

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

print("Training houses:", len(Xh_train))
print("Test houses    :", len(Xh_test))

## 🏋️ Step 18 — Train Linear Regression

Linear Regression finds the best **weights** for this equation:

> price = intercept + w₁ × area + w₂ × bedrooms + w₃ × age

- `coef_` → the weights (how much each feature changes the price)
- `intercept_` → the base price when all features are 0

🎯 Compare what the model **learned** with the **secret formula** we used. They should be close!

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(Xh_train, yh_train)       # TRAIN

learned = pd.DataFrame({
    "Feature": X_h.columns,
    "Learned weight": lin_reg.coef_.round(0),
    "Secret (true) weight": [3000, 200000, -15000],
})
print(f"Learned intercept (base price): ₹{lin_reg.intercept_:,.0f}   (secret: ₹500,000)\n")
learned

**How to read the weights:**
- `area_sqft ≈ 3000` → every extra sq.ft adds about ₹3,000 to the price
- `bedrooms ≈ 2,00,000` → each extra bedroom adds about ₹2 lakh
- `age_years ≈ −15,000` → every extra year of age *reduces* the price by about ₹15,000

The learned weights are **close to** the secret ones but not *exactly* equal, because of the random noise in the data — real data is never perfectly clean.
The model discovered the hidden pattern **just from examples**. That's machine learning! 🎉

## 📏 Step 19 — Evaluate the regression model

| Metric | Meaning | Good value |
|---|---|---|
| **MAE** (Mean Absolute Error) | On average, how many ₹ is the prediction off? | Lower is better |
| **RMSE** (Root Mean Squared Error) | Like MAE but punishes big mistakes more | Lower is better |
| **R² score** | How much of the price variation the model explains | Closer to **1** is better (0 = no better than guessing the average) |

In [ ]:
yh_pred = lin_reg.predict(Xh_test)         # predict prices of the test houses

mae  = mean_absolute_error(yh_test, yh_pred)
rmse = np.sqrt(mean_squared_error(yh_test, yh_pred))
r2   = r2_score(yh_test, yh_pred)

print(f"MAE  : ₹{mae:,.0f}   -> typical error per house")
print(f"RMSE : ₹{rmse:,.0f}")
print(f"R²   : {r2:.3f}   -> model explains {r2:.1%} of price variation")

## 📈 Step 20 — Actual vs Predicted graph

Each dot is one test house. If the model were **perfect**, all dots would sit exactly on the red diagonal line.
The closer the dots are to the line, the better the model.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(yh_test, yh_pred, alpha=0.7)

low, high = yh_test.min(), yh_test.max()
plt.plot([low, high], [low, high], color="red", linestyle="--", label="Perfect prediction")

plt.xlabel("Actual price (₹)")
plt.ylabel("Predicted price (₹)")
plt.title("Actual vs Predicted house prices")
plt.legend()
plt.show()

## 🥊 Step 21 — Compare with Random Forest Regressor

Same 3 steps (create → fit → predict), different model. A Random Forest builds many decision trees and **averages** their predictions.

⚠️ Since our data was created with a straight-line formula, Linear Regression is expected to win here. On messy real-world data with curved patterns, Random Forest often does better. **No model is best for everything** — always compare!

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(Xh_train, yh_train)
rf_pred = rf_reg.predict(Xh_test)

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE (₹)":  [mean_absolute_error(yh_test, yh_pred), mean_absolute_error(yh_test, rf_pred)],
    "RMSE (₹)": [np.sqrt(mean_squared_error(yh_test, yh_pred)), np.sqrt(mean_squared_error(yh_test, rf_pred))],
    "R²":       [r2_score(yh_test, yh_pred), r2_score(yh_test, rf_pred)],
})
comparison.round(3)

## 🏡 Step 22 — Predict the price of a new house

In [ ]:
new_house = pd.DataFrame([[1800, 3, 5]], columns=X_h.columns)   # 1800 sq.ft, 3 bedrooms, 5 years old

predicted_price = lin_reg.predict(new_house)[0]
print(f"🏠 Estimated price: ₹{predicted_price:,.0f}")
print(f"   (give or take about ₹{mae:,.0f}, based on our test error)")

# 🎓 Summary — What you learned

### The supervised learning recipe
```python
# 1. Load data           df = pd.read_csv("data.csv")
# 2. Split X and y       X = df[features];  y = df["target"]
# 3. Train/Test split    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 4. (Scale if needed)   scaler.fit_transform(X_train);  scaler.transform(X_test)
# 5. Create the model    model = SomeModel()
# 6. Train               model.fit(X_train, y_train)
# 7. Predict             y_pred = model.predict(X_test)
# 8. Evaluate            accuracy / confusion matrix  (classification)   |   MAE / RMSE / R²  (regression)
```

### Key ideas to remember
| Concept | One-line meaning |
|---|---|
| Features (X) / Label (y) | Inputs / the answer to predict |
| Classification vs Regression | Predict a category vs predict a number |
| Train / Test split | Practise on one part, get examined on a hidden part |
| Overfitting | Memorised the training data, fails on new data |
| Underfitting | Too simple to catch the pattern |
| Data leakage | Test-set information sneaking into training (always `fit` on train only) |
| Training | Adjust parameters step by step to reduce error (gradient descent) |
| Hyperparameter | A setting *you* choose (like K in KNN) |

### ⚠️ Common beginner mistakes
1. Testing the model on the same data it was trained on
2. Fitting the scaler on the full dataset before splitting (data leakage)
3. Trusting accuracy blindly when classes are imbalanced (e.g. 95% "not fraud") — also check precision and recall
4. Skipping data exploration

---

## 📝 Practice Exercises

1. **Change `test_size`** to 0.3, then 0.1. How does accuracy change?
2. **Remove `stratify=y`** and re-run. Does the class balance in the test set change?
3. In Step 11, **plot a Decision Tree's** accuracy for `max_depth` from 1 to 10. Where does it overfit?
4. Load another built-in dataset — `from sklearn.datasets import load_wine` or `load_breast_cancer` — and repeat the whole classification workflow.
5. In Part 3, **add a new feature** `distance_from_city` to the house data and see how R² changes.
6. Change `learning_rate` in the gradient descent cell to **0.001** and then **0.05**. What happens to `w`? (Hint: try 0.06 too!)

## 🚀 What next?
- **Cross-validation** (`cross_val_score`) → a more reliable test than a single split
- **Hyperparameter tuning** (`GridSearchCV`)
- **Handling categorical columns** (`OneHotEncoder`) and missing values (`SimpleImputer`)
- **Pipelines** to combine scaling + model safely
- **Unsupervised learning** (clustering) and **Deep learning** (neural networks)

*Happy learning! 🚀*